# pivot-ai-handball

Ouvre ce notebook dans le dépôt avec le noyau Python du projet. Dépose ta vidéo dans `data/raw_clips/`, puis indique son nom dans la cellule de configuration. Le notebook cherche la calibration à côté de la vidéo. Les résultats sont écrits dans `data/outputs/`.

## 1. Configurer le match

In [ ]:
import os
from pathlib import Path

import pivot_ai
from pivot_ai.config import chemin_correspondances_video

courant = Path.cwd().resolve()
racine = next(
    (p for p in (courant, *courant.parents) if (p / "scripts" / "build_notebook.py").is_file()),
    None,
)
if racine is None:
    raise RuntimeError("Ouvre le notebook depuis le dépôt pivot-ai-handball.")

CLIPS_DIR = racine / "data" / "raw_clips"
CLIP = str(CLIPS_DIR / "mon_clip.mp4")
CHEMIN_CORRESPONDANCES = chemin_correspondances_video(CLIP)
OUTPUTS_DIR = racine / "data" / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"pivot_ai version : {pivot_ai.__version__}")
print("Clips disponibles :", [p.name for p in sorted(CLIPS_DIR.glob("*.mp4"))])

## 2. Verifier GPU

In [ ]:
import torch
print(f"CUDA dispo : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")

## 3. Choisir un clip et relever les points d'homographie

Affiche une frame du clip. **Survole avec la souris** pour relever les pixels des points caracteristiques du terrain.

**Points a relever (au minimum 4, idealement 6-8 pour activer RANSAC) :**

| Nom | Description | Coords terrain (m) |
|-----|-------------|---------------------|
| A | Coin terrain haut-gauche (proche bas image) | (0, 0) |
| B | Coin terrain haut-droite | (40, 0) |
| C | Coin terrain bas-droite (proche haut image) | (40, 20) |
| D | Coin terrain bas-gauche | (0, 20) |
| E | Ligne mediane cote bas | (20, 0) |
| F | Ligne mediane cote haut | (20, 20) |
| M_prime | Poteau de but gauche, cote bas | (0, 8.5) |
| N_prime | Poteau de but gauche, cote haut | (0, 11.5) |
| M | Poteau de but droit, cote bas | (40, 8.5) |
| N | Poteau de but droit, cote haut | (40, 11.5) |
| K | Sommet demi-cercle 6m gauche | (6, 10) |
| L | Sommet demi-cercle 6m droite | (34, 10) |
| O_prime | Sommet arc 9m gauche | (9, 10) |
| O | Sommet arc 9m droite | (31, 10) |

Voir `pivot_ai/config.py` (POINTS_TERRAIN_M) pour la liste complete.

In [ ]:
import cv2
import plotly.express as px

if not Path(CLIP).is_file():
    raise FileNotFoundError(f"Indique le nom de ton clip dans la cellule de configuration : {CLIP}")

cap = cv2.VideoCapture(CLIP)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
FRAME_REF = total // 2
cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_REF)
ret, frame_ref = cap.read()
cap.release()
if not ret or fps <= 0:
    raise RuntimeError(f"Impossible de lire le clip : {CLIP}")

print(f"Clip : {total} frames @ {fps:.1f} fps ({total/fps:.1f}s)")

fig = px.imshow(cv2.cvtColor(frame_ref, cv2.COLOR_BGR2RGB))
fig.update_layout(width=1300, height=730, dragmode='pan',
                  title=f"Frame {FRAME_REF} — survole pour relever les pixels")
fig.show()

In [ ]:
import json

if not CHEMIN_CORRESPONDANCES.is_file():
    raise FileNotFoundError(
        f"Crée {CHEMIN_CORRESPONDANCES} avec les points relevés sur ce clip. "
        "Utilise python -m scripts.calibrer_terrain avec le chemin du clip depuis la racine du dépôt."
    )

correspondances = json.loads(CHEMIN_CORRESPONDANCES.read_text(encoding="utf-8"))
print(f"Calibration chargée : {len(correspondances)} points")

## 4. Modele de detection

Par defaut le pipeline utilise **YOLO26m COCO** (detecte des 'person' generiques,
tracking fragmente). Si tu as entraine un modele handball via
`notebooks/train_handball_yolo.ipynb`, pointe dessus : il distingue
joueurs / gardiens / arbitres / ballon et reduit fortement la fragmentation.
Si le poids panorama existe dans `models/`, il est sélectionné en priorité ; modifie
`CHEMIN_MODELE_HANDBALL` pour choisir un autre fichier.

In [ ]:
import os
from pivot_ai.config import ModeleConfig

DOSSIER_MODELES = racine / "models"
poids_panorama = DOSSIER_MODELES / "handball_yolo26m_panorama.pt"
poids_standard = DOSSIER_MODELES / "handball_yolo26m.pt"
CHEMIN_MODELE_HANDBALL = str(poids_panorama if poids_panorama.is_file() else poids_standard)

if os.path.exists(CHEMIN_MODELE_HANDBALL):
    modele_config = ModeleConfig.pour_handball(CHEMIN_MODELE_HANDBALL)
    print(f"Modele handball fine-tune : {CHEMIN_MODELE_HANDBALL}")
else:
    modele_config = None  # -> ModeleConfig() par defaut (YOLO26m COCO)
    print("Modele handball absent : fallback YOLO26m COCO (tracking fragmente attendu).")
    print("Entraine-le via notebooks/train_handball_yolo.ipynb pour de meilleurs resultats.")

# Backend de tracking :
#  - "bytetrack": mouvement seul, rapide. MEILLEUR sur plan large broadcast
#    (KPI mesures : moins de reprises d'ID que BoT-SORT sur ce type de source).
#  - "botsort"  : ReID par apparence. Utile seulement si les joueurs sont assez
#    gros a l'image (plan serre) ; sur plan large il degrade (empreintes bruitees).
TRACKER = "bytetrack"
print(f"Tracker : {TRACKER}")

## 5. Test rapide sur clip court

Avant le pipeline complet, on lance une passe rapide pour valider l'enchainement :
- subsample agressif (1 frame sur 5)
- pas de video radar ni decoupage : on veut juste verifier que la chaine tient debout

Sur un clip de 60s a 25fps : ~300 inferences au lieu de 1500.

In [ ]:
from pivot_ai.pipeline import traiter_match_complet
import time

DOSSIER_TEST = f"{OUTPUTS_DIR}/_test_rapide"

t0 = time.time()
resultat_test = traiter_match_complet(
    chemin_video=CLIP,
    correspondances_homographie=correspondances,
    dossier_sortie=DOSSIER_TEST,
    subsample=5,
    generer_video_radar=False,
    decouper_actions=False,
    modele_config=modele_config,
    min_frames_track=10,  # ecarte les fragments de tracking courts
    tracker=TRACKER,
)
duree = time.time() - t0

meta = resultat_test.metadonnees
print(f"\nTest rapide OK en {duree:.1f}s (tracker={meta['tracker']})")
print(f"Pistes totales : {meta['nb_joueurs_avant_filtre']}")
print(f"--- KPI qualite tracking ---")
print(f"Reprises d'ID (switches suspectes) : {meta['nb_reprises_id']}")
print(f"Joueurs estimes (apres recollage)  : {meta['nb_joueurs_estimes']}")
print(f"Longueur moyenne de piste          : {meta['longueur_track_moyenne']} frames")
print(f"Joueurs exploitables (apres filtre): {meta['nb_joueurs_stats']}")
print(f"\nStats (10 premieres lignes) :")
print(resultat_test.stats_joueurs.head(10))

## 6. Pipeline complet

Le pipeline traite une image sur deux avec `subsample=2`.

In [ ]:
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()

t0 = time.time()
resultat = traiter_match_complet(
    chemin_video=CLIP,
    correspondances_homographie=correspondances,
    dossier_sortie=OUTPUTS_DIR,
    subsample=2,
    generer_video_radar=True,
    decouper_actions=True,
    modele_config=modele_config,
    min_frames_track=10,  # ecarte les fragments de tracking courts
    tracker=TRACKER,
)
duree_min = (time.time() - t0) / 60

meta = resultat.metadonnees
print(f"\nPipeline complet termine en {duree_min:.1f} min (tracker={meta['tracker']})")

print("\n=== KPI QUALITE TRACKING ===")
print(f"Pistes totales                     : {meta['nb_joueurs_avant_filtre']}")
print(f"Reprises d'ID (switches suspectes) : {meta['nb_reprises_id']}")
print(f"Joueurs estimes (apres recollage)  : {meta['nb_joueurs_estimes']}")
print(f"Longueur moyenne de piste          : {meta['longueur_track_moyenne']} frames")
print(f"Joueurs exploitables (apres filtre): {meta['nb_joueurs_stats']}")

print("\n=== STATS PAR JOUEUR ===")
print(resultat.stats_joueurs)

print(f"\n=== ACTIONS DETECTEES : {len(resultat.actions_detectees)} ===")
for i, action in enumerate(resultat.actions_detectees):
    print(f"  Action {i+1}: frames {action.frame_debut}-{action.frame_fin} "
          f"({action.duree_s:.1f}s, {action.nb_joueurs_moyen:.1f} joueurs moyens)")

print(f"\n=== CLIPS DECOUPES : {len(resultat.clips_decoupes)} ===")
for clip in resultat.clips_decoupes:
    print(f"  {clip}")

print(f"\n=== ARTEFACTS ===")
print(f"  CSV  : {OUTPUTS_DIR}/stats_joueurs.csv")
print(f"  Parquet : {OUTPUTS_DIR}/stats_joueurs.parquet")
print(f"  Video radar SBS : {resultat.chemin_video_radar}")

## 7. Visualiser quelques frames du rendu final

In [ ]:
import matplotlib.pyplot as plt

video_radar = resultat.chemin_video_radar
cap = cv2.VideoCapture(str(video_radar))
total_radar = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fig, axes = plt.subplots(5, 1, figsize=(20, 22))
for ax, idx in zip(axes, [0, total_radar//4, total_radar//2, 3*total_radar//4, total_radar-1]):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if ret:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Frame {idx}/{total_radar}")
        ax.axis('off')
cap.release()
plt.tight_layout()
plt.show()

## 8. Heatmap d'un joueur

Visualise la densite de presence d'un joueur sur le terrain. On prend par defaut le tracker le plus present.

In [ ]:
import polars as pl
import plotly.express as px
from pivot_ai.stats import generer_heatmap_joueur

# Tracker le plus present dans le clip
stats_tries = resultat.stats_joueurs.sort("nb_frames_detectees", descending=True)
tracker_choisi = int(stats_tries["tracker_id"][0])
print(f"Tracker selectionne : {tracker_choisi} "
      f"({stats_tries['nb_frames_detectees'][0]} frames detectees)")
print(stats_tries.filter(pl.col("tracker_id") == tracker_choisi))

# Heatmap reelle a partir des positions terrain stockees dans ResultatPipeline
positions_joueur = resultat.positions_par_tracker.get(tracker_choisi, [])
heatmap = generer_heatmap_joueur(positions_joueur, resolution=0.5, sigma_lissage=1.5)

fig = px.imshow(
    heatmap,
    origin="lower",
    aspect="equal",
    color_continuous_scale="Hot",
    title=f"Heatmap presence joueur tracker_id={tracker_choisi}",
    labels={"x": "x_terrain (cellules de 0.5m)", "y": "y_terrain (cellules de 0.5m)"},
)
fig.update_layout(width=900, height=500)
fig.show()

## 9. Largeur du bloc defensif au cours du temps

Charge le DataFrame de la largeur du bloc defensif (MHB ou ADV) et le trace via plotly.

In [ ]:
import polars as pl
import plotly.express as px

chemin_bloc_mhb = f"{OUTPUTS_DIR}/largeur_bloc_defensif_mhb.csv"
chemin_bloc_adv = f"{OUTPUTS_DIR}/largeur_bloc_defensif_adv.csv"

if os.path.exists(chemin_bloc_mhb):
    df_mhb = pl.read_csv(chemin_bloc_mhb).with_columns(equipe=pl.lit("MHB"))
    df_adv = pl.read_csv(chemin_bloc_adv).with_columns(equipe=pl.lit("ADV"))
    df = pl.concat([df_mhb, df_adv])
    fig = px.line(
        df.to_pandas(),
        x="temps_s", y="largeur_y_m", color="equipe",
        title="Largeur du bloc defensif au cours du temps",
        labels={"temps_s": "Temps (s)", "largeur_y_m": "Largeur (m)"},
    )
    fig.show()
else:
    print("Pas de fichier largeur_bloc_defensif_*.csv : classification d'equipes a peut-etre echoue (trop peu de joueurs ?)")